# LAASI **Translation** with a Judge for HPC 

***Argonne Argo API*** and ***Ollama Local Server*** Jupyter Notebook version

Code translation source and target scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

nvcc should first be installed via:
sudo apt install nvidia-cuda-toolkit

nvc++ installed via: https://developer.nvidia.com/hpc-sdk

## Setup

In [1]:
from IPython.display import display, Markdown
import requests
import json
import csv
import re
import os
import time
from datetime import datetime
from zoneinfo import ZoneInfo
import subprocess
import shlex  # for splitting command line arguments
import argostream
import ollamastream
from prompt_dictionary import PromptDictionary

# Used in similarity metrics
import difflib
import tokenize  # apparently only defined for Python code
import tiktoken
from io import BytesIO

# Enable the autoreload extension
%load_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1
    
# We need to reload the prompt dictionary because it is under development
%aimport prompt_dictionary
%aimport argostream
    
# Text formatting options
class color:
   PURPLE = '\033[95m'
   CYAN = '\033[96m'
   DARKCYAN = '\033[36m'
   BLUE = '\033[94m'
   GREEN = '\033[92m'
   YELLOW = '\033[93m'
   RED = '\033[91m'
   BOLD = '\033[1m'
   UNDERLINE = '\033[4m'
   END = '\033[0m'

## Pipeline Utility Method Definitions

### LLM Methods

In [2]:
def code_knowledge_llm(llm_source, model, num_ctx, system_prompt, input_context, target_lang, display_response):

    print("\n\n" + color.GREEN + "--> ... " + color.UNDERLINE + "prompt LLM to summarize background knowledge and context." + color.END + "\n")
    
    content_prompt = """
    Before translating the code, please provide a summary description the following background knowledge and context on the programming language.
    """

    if target_lang == "OMP":
        content_prompt = content_prompt + """ 
        Specifically, describe how to implement the omp pragma directive 'target teams' for distributing for loops on a GPU.
        """
        
    content_prompt = content_prompt + """ 
    Only respond with your summary description and avoid repeating the code: 
    """

    content_prompt = content_prompt + input_context
    
    if llm_source == "argo":
        response_text = argostream.argo_llm(model, system_prompt, content_prompt, verifyprompt=False)
    elif llm_source == "lmstudio":
        response_text = lmstudio_llm(model, system_prompt, content_prompt)
    elif llm_source == "ollama":
        response_text = ollamastream.ollama_llm(model, system_prompt, content_prompt, verifyprompt=False)

    print("\n\n----- " + color.BLUE + "CONTEXT KNOWELEDGE SUMMARY" + color.END + " -----")
    print("(generated by the LLM)\n")
    if display_response:
        display(Markdown(response_text))

    return response_text

In [3]:
def code_description_llm(llm_source, model, num_ctx, system_prompt, input_code, display_response):

    print("\n\n" + color.GREEN + "--> ... " + color.UNDERLINE + "prompt LLM to summarize the source code." + color.END + "\n")
    
    content_prompt = """
    Before translating the code, please provide a summary description of the following code. 
    Only respond with your summary description and avoid repeating the code: 
    """
    
    content_prompt = content_prompt + input_code
    
    if llm_source == "argo":
        response_text = argostream.argo_llm(model, system_prompt, content_prompt, verifyprompt=False)
    elif llm_source == "lmstudio":
        response_text = lmstudio_llm(model, system_prompt, content_prompt)
    elif llm_source == "ollama":
        response_text = ollamastream.ollama_llm(model, system_prompt, content_prompt, verifyprompt=False)

    print("\n\n----- " + color.BLUE + "CODE DESCRIPTION" + color.END + " -----")
    print("(generated by the LLM)\n")
    if display_response:
        display(Markdown(response_text))

    return response_text

In [4]:
def code_generator_llm(llm_source, model, num_ctx, system_prompt, content_prompt, display_response):

    print("\n\n" + color.GREEN + "--> ... " + color.UNDERLINE + "prompt LLM to generate translated code." + color.END + "\n")
    
    if llm_source == "argo":
        response_text = argostream.argo_llm(model, system_prompt, content_prompt, verifyprompt=False)
    elif llm_source == "lmstudio":
        response_text = lmstudio_llm(model, system_prompt, content_prompt)
    elif llm_source == "ollama":
        response_text = ollamastream.ollama_llm(model, system_prompt, content_prompt, verifyprompt=False)
        
    print("\n\n----- " + color.BLUE + "GENERATED CODE" + color.END + " -----\n\n")
    if display_response:
        display(Markdown(response_text))

    # Extract only the code block - 
    # Search for the ``` pattern in the text
    #   Regular expression pattern to match text between triple backticks 
    #   that represent code blocks within generated responses from the LLM.
    pattern = r"```(.*?)```"
    match = re.search(pattern, response_text, re.DOTALL)
    # print("RESPONSE TEXT -- ", response_text)
    # print("MATCH -- ", match)
    if match:
        extracted_text = match.group(1)
    else:
        extracted_text = ""
    
    return response_text, extracted_text

### Code Evaluation Methods

In [5]:
def compile_code(source_file, lang, code_compiler, code_compiler_kwds):
       
    # Split the compile keywords into a list of arguments
    compile_kwds_list = shlex.split(code_compiler_kwds)

    if lang == "OMP":
        # The desired name of the compiled program
        output_file = source_file.split('.cpp')[0]
        # Construct the compilation command
        compile_command = [code_compiler] + compile_kwds_list + [source_file, '-o', output_file]
    
    elif lang == "CUDA":
        # The desired name of the compiled program
        output_file = source_file.split('.cu')[0]
        # Construct the compilation command
        compile_command = [code_compiler] + compile_kwds_list + [source_file, '-o', output_file]
        
    print("Compile command: " + str(compile_command))
 
    # Execute the compilation command
    return_result = ""
    try:
        result = subprocess.run(compile_command, capture_output=True, text=True, timeout=120)
    
        # Check if the compilation was successful
        if result.returncode == 0:
            print("\n" + color.BOLD + "Compilation of code successful!" + color.END)
        else:
            # If there was an error, print the error message
            return_result = result.stderr
            print("\n" + color.BOLD + color.RED + "***** Compilation failed." + color.END)
            print(" Error message: ")
            print(return_result)
    except subprocess.TimeoutExpired as e:
        # The child process will have been killed.
        print("\n" + color.BOLD + color.RED + "***** Compilation failed: timeout." + color.END)
        print(f"[ERROR] Compilation timed out after {e.timeout} seconds.")
        # e.stdout and e.stderr contain any partial output
        print("partial stdout:", e.stdout)
        print("partial stderr:", e.stderr)
        return_result = str(e.stdout) + "\n\n" + str(e.stderr)
            
    return return_result, output_file

In [6]:
def execute_code(executable_file, exe_input_parameters):

    # Capture current directory to switch back after code execution
    initial_dir = os.getcwd()

    # Extract the compiled program to execute
    directory_path = os.path.dirname(executable_file)
    executable = os.path.basename(executable_file)

    # Change the working directory to the subdirectory of the executable
    os.chdir(directory_path)

    # Ensure the binary has execute permissions
    subprocess.run(['chmod', '+x', executable])

    # Build execute command
    execute_command = ['./' + executable] + exe_input_parameters
    print("Execute command: " + str(execute_command) + "\n")

    # Execute the binary object with input parameters, if provided
     
    # Ensure the file is open and available before executing
    if os.path.exists(executable):
        try:
            # Open the file to ensure it is accessible
            with open(executable, 'rb') as code_file:
                # Start the process
                start_runtime = time.time()
                process = subprocess.Popen(execute_command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                stdout_lines = []
                stderr_lines = []
                exception_error = ""
                
                # Print the output in real-time
                try:
                    # Read stdout in a separate thread or in a non-blocking way
                    while True:
                        line = process.stdout.readline()
                        if not line:
                            break
                        print(line, end='')  # Print each line as it is received
                        stdout_lines.append(line)
                    process.stdout.close()
            
                    # Read stderr in a separate thread or in a non-blocking way
                    while True:
                        line = process.stderr.readline()
                        if not line:
                            break
                        stderr_lines.append(line)
                    process.stderr.close()
                
                    # Wait for the process to complete and get the return code
                    return_code = process.wait()
                    runtime = time.time() - start_runtime
                    
                    if return_code != 0:
                        error_output = ''.join(stderr_lines)
                        print("Program Execution Error:", error_output)

                except Exception as e:
                    print("An error occurred while executing the program: ", e)
                    exception_error = str(e)
        except IOError as e:
            print(f"Pipeline error | An I/O error occurred: {e.strerror}")
        except Exception as e:
            print(f"Pipeline error | An error occurred: {e}")
    else:
        print(f"Pipeline error | The file {executable} does not exist.")
       
    # Check if the program ran successfully
    return_result = "0"  # default OK
    if return_code == 0:
        print("\n" + color.BOLD + "Program executed successfully!" + color.END)
    else:
        # If there was an error during execution, print the error message
        return_result = "Program exited with return code: " + str(return_code) + " "
        if return_code == -11:
            return_result += "Segmentation fault detected."
        stderr_output = ''.join(stderr_lines)
        # stdout_output = ''.join(stdout_lines)
        # if stdout_output:
        #    return_result += "\nStandard Output: " + stdout_output
        if exception_error != "":
            return_result += "\nException Error: " + exception_error
        if stderr_output:
            return_result += "\nStandard Error: " + stderr_output
        print("\n" + color.BOLD + color.RED + "***** Execution failed." + color.END)
        print("Error message: ")
        print(return_result)
    
    os.chdir(initial_dir)

    stdout_output = ''.join(stdout_lines)

    print("\nLASSI runtime (s): " + str(round(runtime, 4)) + "\n")
    
    return return_result, stdout_output, round(runtime, 4)

In [7]:
def token_count(doc_text):
    tokenizer = tiktoken.get_encoding("cl100k_base")

    encoding = tiktoken.Encoding(
        name="gpt-35-turbo",
        pat_str=tokenizer._pat_str,
        mergeable_ranks=tokenizer._mergeable_ranks,
        special_tokens={
            **tokenizer._special_tokens,
            "<|im_start|>": 100264,
            "<|im_end|>": 100265,
        },
    )

    tokens_doc = encoding.encode(
        doc_text, allowed_special={"<|im_start|>", "<|im_end|>"}
    )

    return len(tokens_doc)
    
def tokenize_by_tiktoken(code):
    # Get an encoding (e.g., "cl100k_base")
    enc = tiktoken.get_encoding("cl100k_base")
    
    # Encode and decode a sample text
    encoded_tokens = enc.encode(code)
    decoded_code = enc.decode(encoded_tokens)

    # Verify that the decoding works correctly
    assert decoded_code == code

    return encoded_tokens

def tokenize_by_tokenize(code):
    # Includes an error handling to "skip" over bad token conversions
    tokens = []
    try:
        readline = BytesIO(code.encode('utf-8')).readline
        token_gen = tokenize.tokenize(readline)
        while True:
            try:
                token = next(token_gen)
                tokens.append(token)
            except IndentationError as e:
                print(f"Indentation error in the code: {e}")
                continue
            except StopIteration:
                break
    except Exception as e:
        print(f"Unexpected error: {e}")

    return tokens

def tokenize_code(code, method="tokenize"):

    if method == "tokenize":
        tokens = tokenize_by_tokenize(code)
    elif method == "tiktoken":
        tokens = tokenize_by_tiktoken(code)
    
    return tokens
    
def token_similarity(code1, code2, method):
    tokens1 = tokenize_code(code1, method)
    tokens2 = tokenize_code(code2, method)
    sequence_matcher = difflib.SequenceMatcher(None, tokens1, tokens2)
    return sequence_matcher.ratio()

def find_line_in_lines(line, lines):
    """Helper function to find a line in a list of lines."""
    for i, l in enumerate(lines):
        if line.strip() == l.strip():
            return i
    return -1

def compare_lines_with_reordering(code1, code2):
    """Compare lines of code1 with lines of code2 allowing reordering."""
    lines1 = code1.splitlines()
    lines2 = code2.splitlines()
    
    matched_lines = 0
    for line1 in lines1:
        index = find_line_in_lines(line1, lines2)
        if index != -1:
            matched_lines += 1
            lines2.pop(index)  # Remove the matched line to prevent duplicate matching
    
    total_lines = max(len(lines1), len(code2.splitlines()))
    similarity_ratio = matched_lines / total_lines if total_lines > 0 else 0
    return similarity_ratio

In [8]:
def llm_judge(llm_source, model, num_ctx, system_prompt, target_lang, source_lang, code_llmjudge_dict, display_response, stage, verifyprompt):

    intent = "code validation judge"

    if target_lang == "OMP":
        target_lang = "OpenMP"

    print("\n\n" + color.GREEN + "--> ... " + color.UNDERLINE + "prompt LLM to judge the validity of the refactored code." + color.END + "\n")
    
    # Use same system prompt.
    # Build user prompt:
    # Fixed instructions first
    # "SOURCE CODE"
    # <insert source code>
    # "STANDARD OUTPUT
    # <insert runtime output>
    # "METRICS"
    # <insert measurements>
    # "REFACTORED CODE"
    # <insert refactored code>
    # "STANDARD OUTPUT
    # <insert runtime output>
    # "METRICS"
    # <insert measurements>
    content_prompt = f"""
    Please compare the difference between the following two codes written in {target_lang} C++.\n
    First, I will provide you first with the SOURCE CODE followed by its STANDARD OUTPUT and METRICS measured during code runtime.\n
    Next, I will provide you with a TRANSLATED CODE followed by its STANDARD OUTPUT and METRICS measured during code runtime. This TRANSLATED CODE was previously refactored by an LLM from {source_lang} C++ to {target_lang} C++.\n
    After you have reviewed both codes and their output, the TRANSLATED CODE is expected (but it may not) to produce equivalent functionality and reasonably similar standard output in its layout and content.\n
    Your task is to strictly evaluate whether the TRANSLATED CODE (1) preserves all intended functionality of the SOURCE CODE and (2) produces printed standard output that is nearly the same as the SOURCE CODE, including the displayed layout and content of the output.\n
    Your careful evaluation should focus mostly on the functional equivalence of the two codes and nearly similar printed standard output of the two codes. Minor differences in numerical values are acceptable in the printed standard output only if they are consistent with the TRANSLATED CODE and these specific values are reasonable and consistent with the SOURCE CODE's intended purpose. Formatting differences are acceptable as long as key details are included that are seen in the standard output of the SOURCE CODE.\n
    Please explain your reasoning, and then conclude your response with VALID if the TRANSLATED CODE is functionally equivalent, or INVALID if it is not.
    """

    content_prompt += f"\n\n# [SOURCE CODE]\n {code_llmjudge_dict['source_code']}"
    content_prompt += f"\n\n## [STANDARD OUTPUT]\n {code_llmjudge_dict['source_stdout']}"
    content_prompt += f"\n\n## [METRICS]\n {code_llmjudge_dict['source_metrics']}"
    if stage == "baselinellm":
        content_prompt += f"\n\n# [TRANSLATED CODE]\n {code_llmjudge_dict['baselinellm_code']}"
        content_prompt += f"\n\n## [STANDARD OUTPUT]\n {code_llmjudge_dict['baselinellm_stdout']}"
        content_prompt += f"\n\n## [METRICS]\n {code_llmjudge_dict['baselinellm_metrics']}"
    elif stage == "lassi_refactored":
        content_prompt += f"\n\n# [TRANSLATED CODE]\n {code_llmjudge_dict['refactored_code']}"
        content_prompt += f"\n\n## [STANDARD OUTPUT]\n {code_llmjudge_dict['refactored_stdout']}"
        content_prompt += f"\n\n## [METRICS]\n {code_llmjudge_dict['refactored_metrics']}"
    
    if llm_source == "argo":
        response_text = argostream.argo_llm(model, system_prompt, content_prompt, verifyprompt=False)
    elif llm_source == "lmstudio":
        response_text = lmstudio_llm(model, system_prompt, content_prompt, 0.2, verifyprompt)
    elif llm_source == "ollama":
        response_text = ollamastream.ollama_llm(model, system_prompt, content_prompt, verifyprompt=False)

    print("\n\n----- " + color.BOLD + color.BLUE + "LLM RESPONSE: " + intent + color.END + " -----\n")
    if display_response:
        display(Markdown(response_text))

    print("\n\n----- " + color.BOLD + color.BLUE + "END LLM RESPONSE" + color.END + " -----\n")

    # look for "VALID" or "INVALID" in the LLM response
    # only return this value
    # NOTE: check "INVALID" first, because "VALID" is a substring
    if "INVALID" in response_text:
        response_validation = "INVALID"
    elif "VALID" in response_text:
        response_validation = "VALID"
    else:
        response_validation = "NONE"
    
    return response_validation, response_text

### The Pipeline

In [9]:
def auto_code_llmgeneration_pipeline(app_folder_name,
                                     this_llm_systemcode_prompt,
                                     this_llm_prompt,
                                     llm_source, 
                                     model,
                                     model_judge,
                                     num_ctx,
                                     use_context,
                                     input_context,
                                     input_code,
                                     target_code,
                                     translated_code_extension,
                                     target_lang,
                                     source_lang,
                                     code_compiler,
                                     code_compiler_kwds,
                                     directory,
                                     file_name,
                                     target_file_path,
                                     display_response,
                                     test_with_error,
                                     generate_test_case,
                                     yes_execute,
                                     exe_input_parameters,
                                     meta_data_file,
                                     code_llmjudge_dict,
                                     trial_number,
                                     experiment_data_directory,
                                     experiment_data_file_object,
                                     experiment_data
                                    ):
    
    # --------------------------------
    # ----- STAGE 0: TEST SOURCE -----
    # --------------------------------
    # Test compile and execute the target souurce code to validate compiler and execution environment
    print("\n\n----- " + color.BLUE + "TARGET SOURCE CODE: Test compile and execute" + color.END + " -----")
    print("\nTest compile original target code: " + str(target_file_path))
    compile_result, execute_file = compile_code(target_file_path, target_lang, code_compiler, code_compiler_kwds)
    if compile_result != "":
        return False
    if yes_execute:
        print("\nTest executing the compiled original target code: " + str(execute_file))
        exe_file_path = execute_file  # directory + "/" + execute_file
        execute_result, exe_output_source, target_runtime = execute_code(exe_file_path, exe_input_parameters)
        if execute_result != "0":
            return False
        else:
            code_llmjudge_dict["source_stdout"] = exe_output_source
            code_llmjudge_dict["source_metrics"] = f"Runtime: {target_runtime} seconds"
    
    print("\n\n")
    now = datetime.now(ZoneInfo("America/Chicago"))
    dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")
    experiment_data["timestamp"] = str(dt_string)
    experiment_data["trial_number"] = trial_number
    experiment_data["target_orig_runtime_s"] = target_runtime

    # Begin LASSI Pipeline Timer
    start_time = time.time()

    general_system_prompt = prompts.get_system_prompt("general_system")
    content = ""

    # --------------------------------------------
    # ----- STAGE 1: BASELINE LLM TRANLATION -----
    # --------------------------------------------
    # Call LLM to translate code without context.
    print("\n\n----- " + color.BLUE + "BASELINE LLM Code Translation: Prompt Strategy Without Context" + color.END + " -----")
    
    content = "Think carefully before developing the following request: "
    content = content + "\n\n" + this_llm_prompt
    content = content + "\n\n" + input_code
    
    # CALL THE LLM
    content = re.sub(' +', ' ', content)
    baselinellm_code_response, baselinellm_code_block = code_generator_llm(llm_source, model, num_ctx, this_llm_systemcode_prompt, content, display_response)

    # SAVE AND EVALUATE THE GENERATED CODE
    if baselinellm_code_block.startswith('c++') or baselinellm_code_block.startswith('cpp'):
        baselinellm_code_block = baselinellm_code_block[3:]
    if baselinellm_code_block.startswith('c'):
        baselinellm_code_block = baselinellm_code_block[1:]    
    
    # SAVE THE GENERATED CODE -- before compiling
    file_path = directory + "/" + file_name + "_baselinellm" + "_Trial" + str(trial_number) + "." + translated_code_extension
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w') as file:
        file.write(baselinellm_code_block)
    file.close()
    
    # Compile the generated code
    baselinellm_compile_result, baselinellm_execute_file = compile_code(file_path, target_lang, code_compiler, code_compiler_kwds)
    print("\nBaseline LLM Execute file: " + str(baselinellm_execute_file))

    if yes_execute and baselinellm_compile_result == "":
        # Unload any previously loaded LLMs in the Ollama server 
        # (clear out the memory)
        if llm_source == "ollama":
            ollamastream.unload_ollama_model(model)
            
        exe_file_path = baselinellm_execute_file  # directory + "/" + execute_file
        baselinellm_execute_result, baselinellm_exe_output, baselinellm_runtime = execute_code(exe_file_path, exe_input_parameters,)
        if baselinellm_execute_result != "0":
            compile_result = "execute_error"  # just a flag
    else:
        # For the baseline LLM case, we do not iterate to fix,
        # So, we set the execute result to a non-empty value.
        baselinellm_exe_output = "NA"
        baselinellm_execute_result = "no executable"
        baselinellm_runtime = 0

    baselinellm_compile_result_data = "Y" if baselinellm_compile_result == "" else "N"
    baselinellm_exe_result_data = "Y" if baselinellm_execute_result == "0" else "N"
    experiment_data["baselinellm_compiled"] = str(baselinellm_compile_result_data)
    experiment_data["baselinellm_execute"] = str(baselinellm_exe_result_data)
    experiment_data["baselinellm_runtime_s"] = baselinellm_runtime
    experiment_data["baselinellm_execute_ratio"] = round(target_runtime/baselinellm_runtime, 4) if baselinellm_runtime != 0 else 0
    
    # LLM-AS-A-JUDGE to verify translated code.
    # This will give us a pass/no pass result to use for calculating a pass@k metric.
    if yes_execute and baselinellm_exe_result_data == "Y":
        code_llmjudge_dict["baselinellm_code"] = baselinellm_code_block
        code_llmjudge_dict["baselinellm_stdout"] = baselinellm_exe_output
        code_llmjudge_dict["baselinellm_metrics"] = f"Runtime: {baselinellm_runtime} seconds"
    
        baselinellm_judge_result, baselinellm_judge_response = llm_judge(llm_source, model_judge, num_ctx, this_llm_systemcode_prompt, target_lang, source_lang, code_llmjudge_dict, display_response, "baselinellm", verifyprompt=False)
        experiment_data["baselinellm_judge_valid"] = str(baselinellm_judge_result)
    else:
        baselinellm_judge_result = "NA"
        baselinellm_judge_response = "NA"
        experiment_data["baselinellm_judge_valid"] = baselinellm_judge_result


    # -----------------------------------------------------
    # ----- STAGE 2: LASSI PIPELINE - CODE TRANLATION -----
    # -----------------------------------------------------
    # Call LLM to translate code without context.
    compile_result = "start"
    content = ""
    execute_result = ""
    error_message = ""
    error_source = ""  # "compiled" or "executed"
    code_block = ""
    self_correction_counter = -1

    # Augmented prompting to handle processing of provided knowledge context
    if use_context:
        content = "[BEGIN CONTEXT KNOWLEDGE] " + input_context + " [END CONTEXT KNOWLEDGE]\n "
    
        llm_code_knowledge_summary_response = code_knowledge_llm(llm_source, model, num_ctx, general_system_prompt, input_context, target_lang, display_response)
        
        content = content + "[BEGIN CONTEXT SUMMARY] " + llm_code_knowledge_summary_response + " [END CONTEXT SUMMARY]\n "
        
        content = content + "Using the above background context with your summarization, "

        llm_code_description_response = code_description_llm(llm_source, model, num_ctx, general_system_prompt, input_code, display_response)
    
        content = content + "Think carefully before developing the following code that you describe as: "

        content = content + llm_code_description_response
    else:
        content = general_system_prompt + "Think carefully before developing the following request: "

    # Save the original context prompt for re-use during error correction.
    original_context_content_prompt = content
    original_prompt_tokens = token_count(original_context_content_prompt)
    
    content = content + " Now, " + this_llm_prompt
    content = content + "\n\n" + input_code
    
    # INITIATE LLM CODE GENERATION ITERATIVE PIPELINE
    while compile_result != "":
        self_correction_counter = self_correction_counter + 1
        
        if compile_result == "start":
            # CALL THE LLM
            content = re.sub(' +', ' ', content)
            # print("\n\n----- " + color.BOLD + "PROMPT" + color.END + " (initial, token count: " + str(original_prompt_tokens) + "): \n" + content + "\n")
            code_response, code_block = code_generator_llm(llm_source, model, num_ctx, this_llm_systemcode_prompt, content, display_response)
        else:
            # CALL THE LLM WITH ERROR CORRECTION
            print("\n\n----- " + color.BOLD + color.RED + "SELF-CORRECTION: " + str(self_correction_counter) + color.END + " -----")
            if compile_result == "execute_error":
                error_correction_prompt_intro = "\n -- The above code was executed after a successul compile with " + code_compiler + " " + code_compiler_kwds + " and produced the following execution error: "
                error_message = execute_result
            else:  # Must be a compiler error
                # NOTE: Maybe do not provide the original code in the other language when trying to fix code in the target language.
                error_correction_prompt_intro = "\n -- The above code was compiled with " + code_compiler + " " + code_compiler_kwds + " and produced the following compile error: "
                error_message = compile_result
            
            # error_correction_prompt_outro = " Eliminate this specific error by re-factoring the above code with an appropriate fix while also using the following background context: " + original_context_content_prompt
            # error_prompt = code_block + error_correction_prompt_intro + error_message + error_correction_prompt_outro
            # error_correction_prompt_outro = " Eliminate this specific error by re-factoring the above code with an appropriate fix."
            error_correction_prompt_outro = " Re-factor the above code with a fix to eliminate the stated error. "
            error_prompt = code_block + error_correction_prompt_intro + error_message + error_correction_prompt_outro
            error_prompt_tokens = token_count(error_prompt)
            print("\n\n----- " + color.BOLD + color.RED + "ERROR SELF-CORRECTION" + color.END + " -----")
            print("\nPROMPT (error correction, token count: " + str(error_prompt_tokens) + "): " + error_prompt + "\n")
            error_prompt = re.sub('\n', '', error_prompt)
            code_response, code_block = code_generator_llm(llm_source, model, num_ctx, this_llm_systemcode_prompt, error_prompt, display_response)

        # SAVE AND EVALUATE THE GENERATED CODE
        if code_block.startswith('c++') or code_block.startswith('cpp'):
            code_block = code_block[3:]
        if code_block.startswith('c'):
            code_block = code_block[1:]    
    
        # SAVE THE GENERATED CODE -- before compiling
        file_path = directory + "/" + file_name + "_lassi" + "_Trial" + str(trial_number) + "." + translated_code_extension
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, 'w') as file:
            file.write(code_block)
        file.close()
    
        # Compile the generated code
        compile_result, execute_file = compile_code(file_path, target_lang, code_compiler, code_compiler_kwds)

        print("\nExecute file: " + str(execute_file))

        if yes_execute and compile_result == "" and self_correction_counter <= 7:
            # Unload any previously loaded LLMs in the Ollama server 
            # (clear out the memory)
            if llm_source == "ollama":
                ollamastream.unload_ollama_model(model)
            
            exe_file_path = execute_file  # directory + "/" + execute_file
            execute_result, exe_output, runtime = execute_code(exe_file_path, exe_input_parameters,)
            if execute_result != "0":
                compile_result = "execute_error"  # just a flag
                
    # end of while pipeline loop

    end_time = time.time() - start_time
    
    print("\n-----" + color.GREEN + "CODE GENERATION COMPLETE" + color.END + "-----\n")
    end_time_minutes = round(end_time/60, 2)
    print("CODE GENERATION TIME: " + str(end_time_minutes) + " minutes.\n")
    print("Self-correction count: " + str(self_correction_counter) + "\n")

    # MEASURE SIMILARITY BETWEEN GENERATED TRANSLATION CODE AND ORIGINAL TARGET CODE
    similarity_tokens = token_similarity(target_code, code_block, "tokenize")
    print(f"Token similarity (tokenize): {similarity_tokens:.2f}")

    similarity_tokens_tiktoken = token_similarity(target_code, code_block, "tiktoken")
    print(f"Token similarity (tiktoken): {similarity_tokens_tiktoken:.2f}")
    
    similarity_lines = compare_lines_with_reordering(target_code, code_block)
    print(f"Line-based similarity with reordering: {similarity_lines:.2f}")

    # LLM-AS-A-JUDGE to verify translated code.
    # This will give us a pass/no pass result to use for calculating a pass@k metric.
    code_llmjudge_dict["refactored_code"] = code_block
    code_llmjudge_dict["refactored_stdout"] = exe_output
    code_llmjudge_dict["refactored_metrics"] = f"Runtime: {runtime} seconds"
    
    judge_result, judge_response = llm_judge(llm_source, model_judge, num_ctx, this_llm_systemcode_prompt, target_lang, source_lang, code_llmjudge_dict, display_response, "lassi_refactored", verifyprompt=False)

    experiment_data["lassi_runtime_s"] = runtime
    experiment_data["lassi_execute_ratio"] = round(target_runtime/runtime, 4)
    experiment_data["lassi_judge_valid"] = str(judge_result)
    experiment_data["lassi_selfcorrections"] = self_correction_counter
    experiment_data["pipeline_runtime"] = str(round(end_time/60, 2))

    meta_data_file += "\n\n"
    meta_data_file += "----- METRICS -----"
    meta_data_file += "\n"
    meta_data_file += "Experiment date-time: " + str(dt_string)
    meta_data_file += "\n"
    meta_data_file += "Time (sec): " + str(round(end_time, 2))
    meta_data_file += "\n"
    meta_data_file += "Experiment trial: " + str(trial_number)
    meta_data_file += "\n"
    meta_data_file += "Source runtime (s): " + str(target_runtime)
    meta_data_file += "\n"
    meta_data_file += "\n----- BASELINE LLM GENERATION -----\n"
    meta_data_file += "\n"
    meta_data_file += "Compile (Y/N): " + baselinellm_compile_result_data
    meta_data_file += "\n"
    meta_data_file += "Execute (Y/N): " + baselinellm_exe_result_data
    meta_data_file += "\n"
    meta_data_file += "Translated runtime (s): " + str(baselinellm_runtime)
    meta_data_file += "\n"
    meta_data_file += "LLM-as-a-Judge result: " + str(baselinellm_judge_result)
    meta_data_file += "\n"
    meta_data_file += "LLM-as-a-Judge reason:\n"
    meta_data_file += "\n----- JUDGE DECISION -----\n"
    meta_data_file += str(baselinellm_judge_response)
    meta_data_file += "\n"
    meta_data_file += "\n----- LASSI GENERATION -----\n"
    meta_data_file += f"Token similarity (tokenize): {similarity_tokens:.2f}"
    meta_data_file += "\n"
    meta_data_file += f"Token similarity (tiktoken): {similarity_tokens_tiktoken:.2f}"
    meta_data_file += "\n"
    meta_data_file += f"Line-based similarity with reordering: {similarity_lines:.2f}"
    meta_data_file += "\n"
    compile_result_data = "Y" if compile_result == "" else "N"
    meta_data_file += "Compile (Y/N): " + compile_result_data
    meta_data_file += "\n"
    exe_result_data = "Y" if execute_result == "0" else "N"
    meta_data_file += "Execute (Y/N): " + exe_result_data
    meta_data_file += "\n"
    meta_data_file += "Self-correction iteration count: " + str(self_correction_counter)
    meta_data_file += "\n"
    meta_data_file += "Translated runtime (s): " + str(runtime)
    meta_data_file += "\n"
    meta_data_file += "LLM-as-a-Judge result: " + str(judge_result)
    meta_data_file += "\n"
    meta_data_file += "LLM-as-a-Judge reason:\n"
    meta_data_file += "\n----- JUDGE DECISION -----\n"
    meta_data_file += str(judge_response)
    if yes_execute:
        meta_data_file += "\n\n\n----- SOURCE: EXECUTION OUTPUT -----\n"
        meta_data_file += exe_output_source
        meta_data_file += "\n\n\n----- BASELINE LLM GENERATED: EXECUTION OUTPUT -----\n"
        meta_data_file += baselinellm_exe_output
        meta_data_file += "\n\n\n----- LASSI GENERATED: EXECUTION OUTPUT -----\n"
        meta_data_file += exe_output

    # Write meta data for this experiment
    filename = directory + "/" + "metadata" + "_" + file_name + "_Trial" + str(trial_number) + "__" + str(dt_string) + ".txt"
    with open(filename, 'w') as file:
        file.write(meta_data_file)
    file.close()

    print("Metadata file: " + str(filename))

    experiment_data["lassi_compiled"] = str(compile_result_data)
    experiment_data["lassi_execute"] = str(exe_result_data)

    # Append collected data to experiment spreadsheet
    print("\nSave experimental data to file.")
    fieldnames = list(experiment_data.keys())
    with open(experiment_data_file_object, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(experiment_data)

    return True

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [10]:
def experimental_setup():
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.

    -- matrix-rotate ["10000", "1"]
    -- jacobi [""] (none required)
    -- layout ["1"]
    -- atomicCost ["1"]
    -- dense-embedding ["10000", "8", "1"]
    -- pathfinder ["10000", "1000", "1000"]
    -- bsearch ["10000", "1"]
    -- entropy ["10000", "1024", "1"]
    -- colorwheel ["10000", "8", "1"]
    -- randomAccess ["1"]
    """
    
    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    app_folder_name = "dense-embedding"

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    source_lang = "CUDA"
    target_lang = "OMP"

    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context = True  # True or False

    # ASSIGN CODE COMPILE COMMAND
    if target_lang == "CUDA":
        code_compiler = "nvcc"
        code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
        # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s  # -arch=sm_89 for RTX 5000
        context_knowledge_selection = "cuda"
    elif target_lang == "OMP":
        # Linux_x86_64 or Linux_aarch64
        # code_compiler = "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # "g++"
        # code_compiler_kwds = "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1"]
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD
        exe_input_parameters = ["10000", "8", "1"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO
    model = "gpt4o"
    model_judge = "gpt41"
    
    # -- FROM OLLAMA
    # model = "qwen3-coder:latest"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 120000  # NOT USED IN THIS VERSION OF LASSI

    # ASSIGN LLM API SOURCE:
    if model in ["gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "gpt41", "gemini25flash"]:
        llm_source = "argo"
    elif "GGUF" in model:
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "01a_judge"  # Assign any label for experiment name.
    display_response = False
    test_with_error = False
    generate_test_case = False

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "timestamp",
        "experiment",
        "app_name",
        "trial_number",
        "llm",
        "llm_judge",
        "source_language",
        "target_language",
        "target_orig_runtime_s",
        "baselinellm_compiled",
        "baselinellm_execute",
        "baselinellm_runtime_s",
        "baselinellm_execute_ratio",
        "baselinellm_judge_valid",
        "lassi_compiled",
        "lassi_execute",
        "lassi_runtime_s",
        "lassi_execute_ratio",
        "lassi_judge_valid",
        "lassi_selfcorrections",
        "pipeline_runtime"
    ]

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        ollamastream.unload_ollama_model(model)

    return (app_folder_name, source_lang, target_lang, model, model_judge, num_ctx, llm_source, use_context,
            context_knowledge_selection, code_compiler, code_compiler_kwds, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, DATAFIELDNAMES)


# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, app_folder_name, source_lang, target_lang):
    filename = app_folder_name + "_" + source_lang.lower() + "2" + target_lang.lower() + "_lassi_pipeline_data.csv"
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [11]:
# Load prompt dictionary (to capture recent manual changes to script, if any)
from prompt_dictionary import PromptDictionary
prompts = PromptDictionary()

all_trials_start = time.time()

# Init code results dict
code_llmjudge_dict = {"source_code": "",
                        "source_stdout": "",
                        "source_metrics": "",
                        "refactored_code": "",
                        "refactored_stdout": "",
                        "refactored_metrics": "",
                    }  # Input values to be used by the LLM-as-a-judge phase

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, source_lang, target_lang, model, model_judge, num_ctx,
    llm_source, use_context, context_knowledge_selection, code_compiler,
    code_compiler_kwds, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, DATAFIELDNAMES
) = experimental_setup()


# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
input_context = prompts.get_codeknowledge(str(context_knowledge_selection))


# DEFINE PROMPT TEMPLATES
system_prompt_selection_index = source_lang + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

codetranslate_prompt_selection_index = source_lang + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_translatecode_prompt = prompts.get_codetranslate_prompt(str(codetranslate_prompt_selection_index))


# LOAD INPUT CODE TO TRANSLATE
input_code_folder = "HeCBench/" + app_folder_name + "/"
input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    input_code_extension = "cu"
source_file_directory = "translated_code/input_codes/" + input_code_folder
source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# LOAD TARGET CODE TO COMPARE TRANSLATION
target_code_filename = app_folder_name + "-" + target_lang.lower() + "_main"
if target_lang == "OMP" or target_lang == "SYCL":
    translated_code_extension = "cpp"
elif target_lang == "CUDA" or target_lang == "HIP":
    translated_code_extension = "cu"
target_file_directory = "translated_code/input_codes/" + input_code_folder
target_file_path = target_file_directory + target_code_filename + "." + translated_code_extension
with open(target_file_path, 'r') as file:
    target_code = file.read()
file.close()

code_llmjudge_dict["source_code"] = target_code

# SPECIFY OUTPUT CODE SAVE
directory = "translated_code/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(codetranslate_prompt_selection_index)
file_name = input_code_filename + "_translatecode_" + target_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "translated_code/" + llm_source + "_" + model + "/experiment/" + experiment + "__" + app_folder_name + "_" + source_lang.lower() + "2" + target_lang.lower() + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES, app_folder_name, source_lang, target_lang)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None

experiment_data["experiment"] = str(experiment) + str(dt_string)
experiment_data["app_name"] = str(app_folder_name)
experiment_data["llm"] = str(model)
experiment_data["llm_judge"] = str(model_judge)
experiment_data["source_language"] = str(source_lang)
experiment_data["target_language"] = str(target_lang)

# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 30

while trial_number <= total_trials:
    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nTranslation: " + system_prompt_selection_index
    meta_data_file += "\nTranslate from: " + input_code_extension
    meta_data_file += "\nTranslate to: " + translated_code_extension
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nJudge Model: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(codetranslate_prompt_selection_index)
    meta_data_file += "\nCode file template: " + directory + "/" + file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    meta_data_file += "\n\nLLM Prompt: " + "\n\n" + this_llm_translatecode_prompt
    
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BOLD + "***** BEGIN PIPELINE *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN PIPELINE
    complete = auto_code_llmgeneration_pipeline(
        app_folder_name,
        this_llm_system_prompt,
        this_llm_translatecode_prompt,
        llm_source, 
        model,
        model_judge,
        num_ctx,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        input_context,
        input_code,
        target_code,
        translated_code_extension,
        target_lang,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        directory,
        file_name,
        target_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        code_llmjudge_dict,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)
    
    trial_number += 1

all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)


***** EXPERIMENT CONFIGURATION SUMMARY *****

Application: dense-embedding

Translation: CUDA_to_OMP
Translate from: cu
Translate to: cpp

LLM Source: argo
Model: gpt4o
Judge Model: gpt41
System Prompt Index: CUDA_to_OMP
Code file template: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge

LLM System Prompt: 

You are a professional coding AI assistant that specializes in translating parallelized CUDA code to C++ code using OpenMP directives. Always provide the complete and fully functional translated code without placeholders, comments, or references suggesting that parts of the original code should be included. Ensure every part of the translated code is explicitly written out. Surround your new generated code with the three characters ```.

LLM Prompt: 

Generate new code to refactor the following parallelized CUDA program to instead use C++ code written with OpenMP directives. To enable GPU offloading, use the `omp pragma` d



----- GENERATED CODE -----


Compile command: ['/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++', '-Wall', '-O3', '-Minfo=accel', '-mp=gpu', '-gpu=cc89', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_baselinellm_Trial1.cpp', '-o', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_baselinellm_Trial1']

Compilation of code successful!

Baseline LLM Execute file: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_baselinellm_Trial1
Execute command: ['./dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_baselinellm_Trial1', '10000', '8', '1']

Number of rows in the embedding table: 10000
Batch size: 8

Embedding dimension: 64
Average execution time of dense embedding kernel (k1): 997.149048 (us)
Average execution time of dense embedding kernel (k2): 52.548004 (us)
PASS

Embedding dimension: 128
Average



----- LLM RESPONSE: code validation judge -----



----- END LLM RESPONSE -----



--> ... prompt LLM to summarize background knowledge and context.


=====< LLM Prompt Not Shown >=====





----- CONTEXT KNOWELEDGE SUMMARY -----
(generated by the LLM)



--> ... prompt LLM to summarize the source code.


=====< LLM Prompt Not Shown >=====





----- CODE DESCRIPTION -----
(generated by the LLM)



--> ... prompt LLM to generate translated code.


=====< LLM Prompt Not Shown >=====





----- GENERATED CODE -----


Compile command: ['/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++', '-Wall', '-O3', '-Minfo=accel', '-mp=gpu', '-gpu=cc89', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial1.cpp', '-o', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial1']

Compilation of code successful!

Execute file: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial1
Execute command: ['./dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial1', '10000', '8', '1']

Program Execution Error: 

***** Execution failed.
Error message: 
Program exited with return code: -6 

LASSI runtime (s): 0.2361



----- SELF-CORRECTION: 1 -----


----- ERROR SELF-CORRECTION -----

PROMPT (error correction, token count: 1444): 
#include <assert.h>
#include <stdio.h>
#include <st

... (193 outputs left out)



----- CODE DESCRIPTION -----
(generated by the LLM)



--> ... prompt LLM to generate translated code.


=====< LLM Prompt Not Shown >=====





----- GENERATED CODE -----


Compile command: ['/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++', '-Wall', '-O3', '-Minfo=accel', '-mp=gpu', '-gpu=cc89', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30.cpp', '-o', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30']

***** Compilation failed: timeout.
[ERROR] Compilation timed out after 120 seconds.
partial stdout: None
partial stderr: None

Execute file: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30


----- SELF-CORRECTION: 1 -----


----- ERROR SELF-CORRECTION -----

PROMPT (error correction, token count: 1459): 
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
#include <chrono>
#include <random>
#include <cmath>
#include <omp.h>

template <typename T>
void reference(
    const T* input,
    const T



----- GENERATED CODE -----


Compile command: ['/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++', '-Wall', '-O3', '-Minfo=accel', '-mp=gpu', '-gpu=cc89', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30.cpp', '-o', 'translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30']

Compilation of code successful!

Execute file: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30
Execute command: ['./dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_lassi_Trial30', '10000', '8', '1']

Number of rows in the embedding table: 10000
Batch size: 8

Embedding dimension: 64
Average execution time of dense embedding kernel (k1): 13270.808594 (us)
Average execution time of dense embedding kernel (k2): 4971.674316 (us)
PASS

Embedding dimension: 128
Average execution time of dense embe



----- LLM RESPONSE: code validation judge -----



----- END LLM RESPONSE -----

Metadata file: translated_code/argo_gpt4o/promptset_CUDA_to_OMP/metadata_dense-embedding-cuda_main_translatecode_OMP_exp01a_judge_Trial30__08-10-2025_t15_00_03.txt

Save experimental data to file.


***** PIPELINE COMPLETED *****




***** ALL TRIALS COMPLETED ***** Total time (min): 83.87


